<a href="https://colab.research.google.com/github/praksb2428-maker/Deep.practice/blob/main/d_l_Project_Final_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **프로젝트 명** : 여름특화 온도 예측 모델

**학번** : 20221703

**학과** : 컴퓨터

**이름** : 박준호

## 1. 프로젝트 소개 및 주제 관련 배경

**프로젝트 소개**

본 프로젝트는 과거 다년도 기상 관측 자료에 내재된 계절적 주기성과 단기 기후 변화 추세를 학습하여, 기상청 관측 데이터가 존재하지 않는 미지의 미래 시점(2026년 이후)이나 특정 타겟 날짜의 평균 기온 및 최고 기온을 정밀하게 추론하는 1D CNN 시계열 시뮬레이션 시스템의 설계 및 구현을 목적으로 합니다.

**주제 관련 배경**

전 세계적인 기후 변동성 심화로 인해 발생하고 있는 한여름철 이상 폭염 현상은 농업 생산성, 국가 전력 수요 관리 등 사회 전반에 걸쳐 치명적인 리스크를 유발하고 있습니다. 그러나 기존의 시계열 연속 전진 예측(Autoregressive) 방식은 장기 시뮬레이션 시 모델이 오차를 줄이기 위해 지루한 평균값으로 수렴(수치 마비 현상)하거나, 데이터 공백기(겨울철)를 거치며 기온이 비정상적으로 하락하는 치명적인 한계를 지니고 있었습니다. 이에 본 연구는 특정 타겟 기일의 전후 기후 파동을 직접 매칭하는 새로운 아키텍처를 도입하여 미래 기온 예측의 독립성과 정밀도를 극대화하고자 하였습니다.


## 2. 데이터셋 소개 (Dataset Specification)

- 원본 데이터: 2010-2025 Summer Temperature Data.xlsx (16개년 엑셀 관측치)

- 최종 데이터: processed_summer_weather_data.csv (정제 완료 파일)

- 시계열 범위: 매년 여름철 집중 영역 (5월, 6월, 7월, 8월)

- 컬럼 명세:

1. Date: 기준 날짜 (YYYY-MM-DD, 시계열 인덱스축)

2. Average Temperature(℃): 일 평균기온 (연속형 타겟 변수 1)

3. Maximum(℃): 일 최고기온 (연속형 타겟 변수 2)

4. Precipitation(mm): 강수량 (연속형 입력 특징 변수)

5. Avg_Temp_Level / Max_Temp_Level / Precip_Level: 기온 및 강수 구간별 정수형 등급 가공 변수

## 3. 전처리 과정 (Data Preprocessing)

- 날짜 포맷 정립: Date 컬럼을 판다스 시계열 타입(datetime64)으로 변환 후 시계열 인덱스로 설정 및 순방향 정렬.

- 강수량 결측치 제어: 기상 관측 기록상 비가 오지 않아 공백(Null)으로 방치되어 있던 대량의 비 결측 구역을 수치 연산이 가능하도록 강수량 0.0값으로 완벽히 치환.

- 파생 변수 생성: 연속형 기온 및 강수량 수치 데이터를 파이썬 연산을 통해 레벨화하여 문맥 인지 성능을 돕는 범주형 등급 변수(Avg_Temp_Level, Max_Temp_Level, Precip_Level)를 자동 생성 및 병합.

- 이상치 방어 스케일링 (RobustScaler): 한국 여름철 특유의 기록적인 기습 폭염 및 게릴라성 폭우 등 극단적 아웃라이어에 의해 모델의 중심축이 무너지지 않도록, 평균 대신 중앙값(Median)과 IQR(사분위수 간 범위)을 기준 축으로 잡는 RobustScaler로 전처리 사양을 최종 고정.

- 시퀀스 버퍼 형성: 과거 연속된 7일간의 다변량 기상 흐름 구조를 추출하여 하나의 3차원 텐서 데이터 규격(WINDOW_SIZE = 7)으로 빌드 후 딥러닝 모델에 인계.

## 4. 모델 구조 (Model Architecture)

- 특징 추출 (Conv1D): Conv1D(filters=64, kernel_size=3, activation='relu')를 활용해 7일 버퍼 내부의 단기 국소 기후 파동(로컬 패턴)을 정밀 추출.

- 과적합 강력 규제: 파라미터 폭발로 노이즈 암기가 일어나기 쉬운 Flatten() 직후 공간으로 Dropout(0.2)을 전방 배치하여 모델 일반화 성능 극대화 (기존 출력층 직전 배치 오류 보정).

- 차원 보존 최적화: 7일 미만의 짧은 버퍼 환경에서 시간 축 정보 파괴를 유발하던 MaxPooling1D 레이어를 물리적으로 완전 삭제하여 특징 맵 손실 차단.

- 최종 출력단: 일 평균기온과 일 최고기온을 동시에 도출하는 Dense(2) 다중 출력(Multi-output) 구조.

## 5. 레퍼런스 개선점 (Reference & Optimization)

- 단순 1D CNN 단일 구조 선회 (모델 경량화): 최초 기획안의 "CNN-LSTM" 복합 구조는 7일 단기 버퍼 환경에서 파라미터가 비대해져 연산 자원 소모가 심하고 속도가 과도하게 무거워지는 문제가 발생함. 이에 자원 소모가 큰 순환 신경망 계열(LSTM, GRU)을 전면 제외하고, 가볍고 빠른 가동이 가능하면서 국소 특징 추출 성능이 우수한 경량형 단순 1D CNN 단일 아키텍처로 선회하여 시스템 최적화 달성.

- 데이터 누수(Data Leakage) 원천 차단: 시간의 순방향성을 완벽히 고수하며 순차적으로 검증 구역을 늘려나가는 5-Fold 시계열 교차 검증(TimeSeriesSplit) 체계 확립.

- 안정성 추적 엔진 내장 (Stability Track): 폴드별 오차 지표(MAE, RMSE, R², MAPE) 산출과 함께, 교차검증 완료 시 5개 폴드의 성능 변화 안정성을 한눈에 모니터링할 수 있는 2x2 격자형 선형 추이 시각화 그래프(Stability Track) 코드를 파이프라인 내에 완전히 내장.

## 6. 프로젝트 결과 (Project Results)

- 실전 추론 오차 수준: 딥러닝 예측 모델과 사후 제어 필터를 종합 가동하여 미래 기온을 시뮬레이션한 결과, 실제 관측된 기온 대비 일 최고기온 및 일 평균기온에서 약 3℃ ~ 4℃ 내외의 안정적인 오차 범위를 기록하며 실전 예측 유효성을 증명함.

- 하이브리드 사후 제어: 5-8월 이외 날짜 인입 시 통계값으로 대체하는 우회 제어(Fallback Clamping), 과거 14개년 동일 기일의 상위 65% 백분위수 데이터 합성, 그리고 한여름철(7-8월) 폭염 가중치(평균 +1.3℃, 최고 +2.6℃) 보정을 연동함.

- 시스템 결정론 확보 (Determinism): 입력 날짜의 연·월·일 조합으로 고유한 정수형 시드 키(seed_key)를 생성 및 재고정하는 로직을 구축하여, 실행 마다 결과가 요동치는 무작위성 문제를 해결하고 동일 날짜 조회 시 항상 일치하는 고정 예측값을 도출함.

## 7. 추후 발전 방향 (Future Work)

- 기상 인자 확장: 현재의 단일 관측소 기반 기상 데이터를 넘어 인근 지역의 해수면 온도(SST), 기압 배치도, 습도 데이터 등 다각도 기후 인자를 추가 확보하여 인풋 특징 벡터 고도화.

- 실시간 스트리밍 파이프라인 개설: 고정된 csv 로드 방식에서 탈출하여 기상청 OpenAPI 연동을 통해 실전 추론 시점의 최신 7일 데이터가 모델 버퍼에 즉각 스냅샷 형태로 피딩되는 실시간 시뮬레이션 인터페이스로의 전환 도모.